# Oil-Well-Optimization

# 1. Descarga y prepara los datos. Explica el procedimiento.

In [94]:
#Importar las librerías

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.utils import shuffle
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

In [95]:
#Cargar los datos
geo_data_0 = pd.read_csv('/datasets/geo_data_0.csv')
geo_data_1 = pd.read_csv('/datasets/geo_data_1.csv')
geo_data_2 = pd.read_csv('/datasets/geo_data_2.csv')


# 2. Entrena y prueba el modelo para cada región en geo_data_0.csv

### 2.1. Divide los datos en un conjunto de entrenamiento y un conjunto de validación en una proporción de 75:25

In [96]:
#Análisis exploratorio de datos del DF data_0
print(geo_data_0.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None


In [97]:
#Dividir los datos en entrenamiento y validación

# Definir características y objetivo
features = geo_data_0[['f0', 'f1', 'f2']]
target = geo_data_0['product']

# Dividir los datos (75% entrenamiento, 25% validación)
features_train, features_valid, target_train, target_valid = train_test_split(
    features,
    target,
    test_size=0.25, #Siempre se señala el % de validación
    random_state=12345
)

# Verificar tamaños
print(features_train.shape)
print(features_valid.shape)


(75000, 3)
(25000, 3)


### 2.2 Entrena el modelo y haz predicciones para el conjunto de validación.

In [98]:
#Entrenar el modelo con regresión logística

model = LinearRegression()

model.fit(features_train, target_train)

LinearRegression()

### 2.3 Guarda las predicciones y las respuestas correctas para el conjunto de validación.

In [99]:
#Hacer predicciones
predictions_valid = model.predict(features_valid)


# Crear un DataFrame con predicciones y valores reales
results_valid = pd.DataFrame({
    'predicted_product': predictions_valid,
    'actual_product': target_valid.reset_index(drop=True)
})

# Mostrar las primeras filas
results_valid.head()

,predicted_product,actual_product
0,95.894952,10.038645
1,77.572583,114.551489
2,77.892640,132.603635
3,90.175134,169.072125
4,70.510088,122.325180


### 2.4 Muestra el volumen medio de reservas predicho y RMSE del modelo.

In [100]:
#Calcular el volumen medio de reservas predicho

mean_volume = results_valid['predicted_product'].mean()
print('El volumen medio de reservas predicho es de:', mean_volume)

#Calcular el RMSE
rmse = mean_squared_error(results_valid['actual_product'], results_valid['predicted_product'], squared=False) 
#Si se usa el parámetro square=False, se obtiene el rmse directamente. Si se usa el parámentro square=True, se obtiene el mse, es decir, el cuadrado del rmse.
print('El RMSE, o Raíz del Error Cuadrático Medio, es:', rmse)


El volumen medio de reservas predicho es de: 92.59256778438035
El RMSE, o Raíz del Error Cuadrático Medio, es: 37.5794217150813


### 2.5 Analiza los resultados

El modelo de regresión lineal predice un volumen medio de reservas cercano a 92 mil barriles por pozo, con un RMSE aproximado de 37 mil barriles. Aunque el error relativo es considerable, el modelo resulta adecuado para la selección comparativa de pozos. La incertidumbre observada justifica la necesidad de un análisis de riesgo mediante bootstrapping antes de tomar una decisión de inversión.

### 2.6 Coloca todos los pasos previos en funciones, realiza y ejecuta los pasos 2.1-2.5 para los archivos 'geo_data_1.csv' y 'geo_data_2.csv'.

In [113]:
def volume(data):
    
#Definir los features y el target
    features = data[['f0', 'f1', 'f2']]
    target = data['product']

#Separar el DF en conjunto de validación y entrenamiento 
    features_train, features_valid, target_train, target_valid = train_test_split(
    features,
    target,
    test_size=0.25,
    random_state=12345
    )
    
#Selccionear el modelo y entrenarlo
    model = LinearRegression()
    model.fit(features_train, target_train)

#Calcular las predicciones y los features en un solo DF
    predictions_valid = model.predict(features_valid)
    results_valid = pd.DataFrame({
    'predicted_product': predictions_valid,
    'actual_product': target_valid.reset_index(drop=True)
    })

#Calcular el volumen promedio y el RMSE
    mean_volume = results_valid['predicted_product'].mean()
    print('El volumen medio de reservas predicho es de:', mean_volume)
    rmse = mean_squared_error(results_valid['actual_product'], results_valid['predicted_product'], squared=False) 
    print('El RMSE, o Raíz del Error Cuadrático Medio, es:', rmse)
    return results_valid, mean_volume, rmse


In [102]:
#Para el DF geo_data_1:

volume(geo_data_1)

El volumen medio de reservas predicho es de: 68.728546895446
El RMSE, o Raíz del Error Cuadrático Medio, es: 0.893099286775617


(       predicted_product  actual_product
 0              82.663314       80.859783
 1              54.431786       53.906522
 2              29.748760       30.132364
 3              53.552133       53.906522
 4               1.243856        0.000000
 ...                  ...             ...
 24995         136.869211      137.945408
 24996         110.693465      110.992147
 24997         137.879341      137.945408
 24998          83.761966       84.038886
 24999          53.958466       53.906522
 
 [25000 rows x 2 columns],
 68.728546895446,
 0.893099286775617)

In [103]:
#Para el Df geo_data_2:

volume(geo_data_2)

El volumen medio de reservas predicho es de: 94.96504596800489
El RMSE, o Raíz del Error Cuadrático Medio, es: 40.02970873393434


(       predicted_product  actual_product
 0              93.599633       61.212375
 1              75.105159       41.850118
 2              90.066809       57.776581
 3             105.162375      100.053761
 4             115.303310      109.897122
 ...                  ...             ...
 24995          78.765887       28.492402
 24996          95.603394       21.431303
 24997          99.407281      125.487229
 24998          77.779912       99.422903
 24999         129.032417      127.445075
 
 [25000 rows x 2 columns],
 94.96504596800489,
 40.02970873393434)

### Análisis

En geo_data_0, el modelo predice un volumen medio de reservas de aproximadamente 92.6 mil barriles, con un RMSE de aproximadamente 37.6, lo que indica un buen potencial productivo, aunque con una incertidumbre considerable.

Para geo_data_1, el volumen medio es menor (aproximadamente 68.7 mil barriles), pero el RMSE muy bajo (aproximadamente 0.89) muestra una alta precisión en las predicciones, sugiriendo bajo riesgo, aunque con menor beneficio esperado.

En geo_data_2, se obtiene el mayor volumen medio predicho (aproximadamente 95.0 mil barriles), pero también la mayor incertidumbre (RMSE aproximadamente 40.0), lo que implica un mayor riesgo.

# 3.Cálculo de ganancias

### 3.1 Almacena todos los valores necesarios para los cálculos en variables separadas.

In [112]:
#Constantes

# Puntos explorados
N_POINTS = 500       

# Pozos a desarrollar
N_WELLS = 200           

# Presupuesto en USD
BUDGET = 100_000_000    

# USD por mil barriles
REVENUE_PER_UNIT = 4500 


results_0, mean_0, rmse_0 = volume(geo_data_0)
results_1, mean_1, rmse_1 = volume(geo_data_1)
results_2, mean_2, rmse_2 = volume(geo_data_2)

#Datos de valores predichos y valores reales para geo_data_0
predictions_0 = results_0['predicted_product']
actual_0 = results_0['actual_product']

#Datos de valores predichos y valores reales para geo_data_1
predictions_1 = results_1['predicted_product']
actual_1 = results_1['actual_product']

#Datos de valores predichos y valores reales para geo_data_2
predictions_2 = results_2['predicted_product']
actual_2 = results_2['actual_product']




El volumen medio de reservas predicho es de: 92.59256778438035
El RMSE, o Raíz del Error Cuadrático Medio, es: 37.5794217150813
El volumen medio de reservas predicho es de: 68.728546895446
El RMSE, o Raíz del Error Cuadrático Medio, es: 0.893099286775617
El volumen medio de reservas predicho es de: 94.96504596800489
El RMSE, o Raíz del Error Cuadrático Medio, es: 40.02970873393434


### 3.2 Dada la inversión de 100 millones por 200 pozos petrolíferos, de media un pozo petrolífero debe producir al menos un valor de 500,000 dólares en unidades para evitar pérdidas (esto es equivalente a 111.1 unidades). Compara esta cantidad con la cantidad media de reservas en cada región.

El volumen medio de reservas predicho en todas las regiones se sitúa por debajo del umbral de rentabilidad de 111.1 unidades por pozo, lo que indica que la rentabilidad solo puede lograrse mediante la selección de los pozos con mayor potencial productivo.

### 3.3 Presenta conclusiones sobre cómo preparar el paso para calcular el beneficio

Antes de calcular el beneficio económico, fue necesario preparar y estructurar adecuadamente los datos. Se entrenó un modelo de regresión lineal para cada región y se obtuvieron predicciones del volumen de reservas para pozos no utilizados en el entrenamiento, lo que permite simular escenarios reales de exploración.

Se almacenaron por separado las predicciones del modelo y los valores reales de reservas, ya que las primeras se utilizan para seleccionar los 200 pozos con mayor potencial, mientras que los valores reales se emplean para calcular el beneficio efectivo. Asimismo, se definieron y fijaron las constantes del proyecto (presupuesto, número de pozos e ingreso por unidad), garantizando coherencia y reproducibilidad en los cálculos posteriores.

Finalmente, al observar que el volumen medio de reservas en todas las regiones es inferior al umbral de rentabilidad por pozo, se concluye que la rentabilidad depende de una selección óptima de pozos, lo que justifica la aplicación del análisis de ganancias y riesgo mediante bootstrapping como siguiente etapa del proyecto.

# 4. Escribe una función para calcular la ganancia de un conjunto de pozos de petróleo seleccionados y modela las predicciones:

### 4.1 Elige los 200 pozos con los valores de predicción más altos de cada una de las 3 regiones (es decir, archivos 'csv')

In [105]:
#Para la región 0:
top_200_0 = results_0.sort_values(
    by='predicted_product',
    ascending=False
).head(200)

#Para la región 1:
top_200_1 = results_1.sort_values(
    by='predicted_product',
    ascending=False
).head(200)

#Para la región 2:
top_200_2 = results_2.sort_values(
    by='predicted_product',
    ascending=False
).head(200)

print(top_200_0)

       predicted_product  actual_product
9317          180.180713      162.810993
219           176.252213      153.639837
10015         175.850623      162.153488
11584         175.658429       96.893581
23388         173.299686      178.879516
...                  ...             ...
7888          148.507064      179.683422
7890          148.481767       95.396917
24051         148.476498      160.361464
24160         148.436761      102.186603
20340         148.365941      119.890261

[200 rows x 2 columns]


### 4.2 Resume el volumen objetivo de reservas según dichas predicciones. Almacena las predicciones para los 200 pozos para cada una de las 3 regiones.

In [106]:
#Calcular las reservas predichas por el modelo para cada región:
target_volume_0 = top_200_0['predicted_product'].sum()
target_volume_1 = top_200_1['predicted_product'].sum()
target_volume_2 = top_200_2['predicted_product'].sum()

print('Volumen objetivo – Región 0:', target_volume_0)
print('Volumen objetivo – Región 1:', target_volume_1)
print('Volumen objetivo – Región 2:', target_volume_2)

Volumen objetivo – Región 0: 31102.3308388114
Volumen objetivo – Región 1: 27746.026782163426
Volumen objetivo – Región 2: 29603.898658318347


### 4.3 Calcula la ganancia potencial de los 200 pozos principales por región. Presenta tus conclusiones: propón una región para el desarrollo de pozos petrolíferos y justifica tu elección.

In [107]:
#Para la región 0
potential_gains_0 = target_volume_0 * 4500 - BUDGET

#Para la región 1
potential_gains_1 = target_volume_1 * 4500 - BUDGET

#Para la región 2
potential_gains_2 = target_volume_1 * 4500 - BUDGET


print(f'La ganancia potencial de los 200 pozos principales para la región 0 es de: {potential_gains_0} USD')
print(f'La ganancia potencial de los 200 pozos principales para la región 0 es de: {potential_gains_1} USD')
print(f'La ganancia potencial de los 200 pozos principales para la región 0 es de: {potential_gains_2} USD')


La ganancia potencial de los 200 pozos principales para la región 0 es de: 39960488.77465129 USD
La ganancia potencial de los 200 pozos principales para la región 0 es de: 24857120.51973541 USD
La ganancia potencial de los 200 pozos principales para la región 0 es de: 24857120.51973541 USD


# Calcula riesgos y ganancias para cada región

### 5.1 Utilizando las predicciones que almacenaste en el paso 4.2, emplea la técnica del bootstrapping con 1000 muestras para hallar la distribución de los beneficios

In [108]:
#Seleccionar los 200 mejores pozos dentro de cada muestra
def calculate_profit(sample):
    top_200 = sample.sort_values(
        by='predicted_product',
        ascending=False
    ).head(200)

    total_volume = top_200['actual_product'].sum()
    profit = total_volume * 4500 - 100_000_000

    return profit

#Crear una función para aplicar bootstrapping

def bootstrap_profit(results, n_samples=1000):
    profits = []

    for _ in range(n_samples):
        sample = results.sample(
            n=500,
            replace=True
        )

        profit = calculate_profit(sample)
        profits.append(profit)

    return np.array(profits)


In [109]:
#Usar la función para la región 0:
profits_0 = bootstrap_profit(results_0)

#Usar la función para la región 1:
profits_1 = bootstrap_profit(results_1)

#Usar la función para la región 2:
profits_2 = bootstrap_profit(results_2)


### Encuentra el beneficio promedio, el intervalo de confianza del 95% y el riesgo de pérdidas. La pérdida es una ganancia negativa, calcúlala como una probabilidad y luego exprésala como un porcentaje

In [110]:
#Crear una función para calcular el beneficio promedio, el intervalo de confianza y el riesgo de pérdidas
def analyze_profits(profits):
    mean_profit = profits.mean()

    lower = np.percentile(profits, 2.5)
    upper = np.percentile(profits, 97.5)

    risk = (profits < 0).mean() * 100  # porcentaje

    return mean_profit, lower, upper, risk


In [111]:
#Usar la función para cada región

mean_0, low_0, high_0, risk_0 = analyze_profits(profits_0)
mean_1, low_1, high_1, risk_1 = analyze_profits(profits_1)
mean_2, low_2, high_2, risk_2 = analyze_profits(profits_2)

print('Región 0')
print(f'Beneficio promedio: {mean_0:,.2f}')
print(f'Intervalo de Confianza del 95%: [{low_0:,.2f}, {high_0:,.2f}]')
print(f'Riesgo de pérdidas: {risk_0:.2f}%\n')

print('Región 1')
print(f'Beneficio promedio: {mean_1:,.2f}')
print(f'Intervalo de Confianza del 95%: [{low_1:,.2f}, {high_1:,.2f}]')
print(f'Riesgo de pérdidas: {risk_1:.2f}%\n')

print('Región 2')
print(f'Beneficio promedio: {mean_2:,.2f}')
print(f'Intervalo de Confianza del 95%: [{low_2:,.2f}, {high_2:,.2f}]')
print(f'Riesgo de pérdidas: {risk_2:.2f}%')


Región 0
Beneficio promedio: 3,835,193.43
Intervalo de Confianza del 95%: [-1,155,800.17, 8,636,449.71]
Riesgo de pérdidas: 6.00%

Región 1
Beneficio promedio: 4,435,415.71
Intervalo de Confianza del 95%: [389,722.11, 8,515,259.51]
Riesgo de pérdidas: 1.10%

Región 2
Beneficio promedio: 3,890,432.33
Intervalo de Confianza del 95%: [-1,470,714.68, 8,818,520.84]
Riesgo de pérdidas: 6.80%


### 5.3 Presenta tus conclusiones: propón una región para el desarrollo de pozos petrolíferos y justifica tu elección. ¿Coincide tu elección con la elección anterior en el punto 4.3?


Tras analizar las tres regiones mediante modelos de regresión lineal y evaluar los beneficios potenciales utilizando la técnica de bootstrapping, se concluye que la región 1 es la opción más adecuada para el desarrollo de nuevos pozos petrolíferos.

Aunque en un análisis preliminar la región 0 mostraba una ganancia potencial elevada basada en la selección directa de los 200 pozos con mayores predicciones, el análisis de riesgo reveló que esta región presenta un riesgo de pérdidas del 6.0%, superior al umbral máximo permitido del 2.5%, así como un intervalo de confianza con valores negativos. Por este motivo, la región 0 no cumple con los criterios de viabilidad establecidos en el proyecto.

En contraste, la región 1 combina el mayor beneficio promedio esperado entre las regiones viables con un riesgo de pérdidas del 1.1%, claramente inferior al límite establecido. Además, su intervalo de confianza del 95% se mantiene completamente en valores positivos, lo que indica una mayor estabilidad y menor incertidumbre en los escenarios simulados.

Por tanto, la elección final se fundamenta no solo en la maximización del beneficio esperado, sino también en una evaluación rigurosa del riesgo, lo que refuerza la solidez de la decisión y demuestra la importancia del uso de técnicas estadísticas como el bootstrapping para la toma de decisiones de inversión.